# Constitutional AI: A Per-Rule Critique-and-Revise Loop

> **What you'll learn:** How to implement **Constitutional AI** as a critique loop — an explicit, named set of principles ("a constitution") that a draft response is checked against, rule by rule, with a structured pass/fail verdict and a targeted revision for every rule that fails.

Constitutional AI (Anthropic, 2022) trains/aligns a model against a small set of written principles instead of relying only on implicit human preference labels. The core *mechanism* — draft, critique against each principle, revise, repeat — is exactly the **Reflection** pattern from this folder, just with the critique broken into named, auditable rules instead of one freeform review.

This notebook implements that mechanism end to end:

1. Define a small, explicit **constitution** — a list of named rules.
2. **Draft** a response to a prompt chosen to plausibly break at least one rule.
3. **Critique** the draft against *each rule independently*, with a structured `{rule, passes, critique}` verdict per rule.
4. **Revise** the draft to address every failing rule, and repeat the critique — bounded to a small number of iterations.
5. Inspect the before/after response and the full table of per-rule verdicts across iterations.

> **Scope note — read this first.** This is a **teaching-style** critique loop meant to make the Constitutional AI mechanism concrete and inspectable. It is **not** a production safety system: there's no adversarial red-teaming, no calibrated judge model, no logging/monitoring, and no policy sign-off process. Production-grade guardrails, content moderation, and safety policy live in [`12_Production_and_Observability/Safety_and_Alignment/`](../../../12_Production_and_Observability/Safety_and_Alignment/) — this notebook should not be mistaken for that.


### What we are going to do

- Write a **constitution**: ~5 named rules as plain Python data (name, rule text) — no framework magic, just a list we can iterate over.
- Ask the LLM a question that *invites* overconfident, speculative medical advice — a good stress test for a "no professional advice" / "acknowledge uncertainty" constitution.
- Generate a **draft** response with no constitution applied at all, so it plausibly fails at least one rule.
- Run a **structured critique**: one LLM call that returns a list of `{rule, passes, critique}` verdicts, one entry per rule, using a Pydantic schema so the output is a typed list rather than freeform text.
- If any rule fails, **revise**: feed the draft plus every failing critique back to the LLM and ask for a rewrite that addresses each one, then critique the revision again.
- Bound the loop to **2 iterations** so it terminates even if a rule keeps failing, and print the full verdict history so we can see exactly what changed and why.


### Setting up the Environment

We reuse the repo's shared LLM factory (`helpers.get_llm`), which auto-selects a provider/model based on platform (Groq on Windows, Databricks on macOS). See the root `CLAUDE.md` for the required `.env` variables (`GROQ_API_KEY` on Windows, Databricks credentials on macOS).


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Import the LLM factory and initialize the language model
# Input   : helpers.get_llm() — platform-aware factory (Groq on Windows, Databricks on macOS)
# Output  : `llm` — configured LLM instance
# ─────────────────────────────────────────────

from helpers import get_llm

llm = get_llm()
print("LLM initialized.")


### Step 1 — Define the Constitution

A constitution here is nothing more than a small, human-readable list of named rules. Keeping it as plain data (rather than baking it into one giant prompt string) is what lets us critique **each rule independently** in Step 3, instead of getting one blended "looks fine" judgment.


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Define the constitution as a list of named, human-readable rules
# Input   : None
# Output  : `CONSTITUTION` — list of {"name", "rule"} dicts
# ─────────────────────────────────────────────

CONSTITUTION = [
    {
        "name": "Helpful and Direct",
        "rule": "Be helpful and directly answer the question the user actually asked.",
    },
    {
        "name": "No Professional Advice",
        "rule": (
            "Do not give specific medical, legal, or financial advice (e.g. a diagnosis, "
            "a treatment plan, a legal conclusion, or a specific financial recommendation). "
            "Instead, suggest consulting a qualified professional for anything specific to "
            "the user's situation."
        ),
    },
    {
        "name": "No Speculation as Fact",
        "rule": (
            "Avoid presenting speculation, guesses, or unverified possibilities as if they "
            "were established fact."
        ),
    },
    {
        "name": "Be Concise",
        "rule": "Keep the response tight and focused — no padding, hedging filler, or repetition.",
    },
    {
        "name": "Acknowledge Uncertainty",
        "rule": (
            "Explicitly acknowledge where the answer is uncertain or depends on details "
            "the user hasn't provided."
        ),
    },
]

for item in CONSTITUTION:
    print(f"- {item['name']}: {item['rule']}")


### Step 2 — Draft Response

We pick a prompt that *invites* the model to violate this constitution on the first try: a health question phrased to pull for a confident diagnosis and treatment plan. We deliberately generate the draft with **no constitution in the prompt at all**, so the critique loop in the next steps has real work to do.


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Generate an initial (unconstrained) draft response
# Input   : USER_PROMPT — a question that invites overconfident medical speculation
# Output  : `draft_response` — the first-draft answer text
# ─────────────────────────────────────────────

USER_PROMPT = (
    "My knee has been hurting for two weeks after I started running again. "
    "What's wrong with it and exactly how should I treat it?"
)

draft_prompt = (
    "Answer the user's question as helpfully as you can.\n\n"
    f"User question: {USER_PROMPT}"
)

draft_response = llm.invoke(draft_prompt).content

print("=== DRAFT RESPONSE ===")
print(draft_response)


### Step 3 — Critique Against Every Rule

Instead of one freeform "is this good?" critique, we ask for a **structured verdict per rule**. `RuleVerdict` captures exactly one rule's outcome (`rule`, `passes`, `critique`); `CritiqueReport` wraps a `verdicts` list, one entry per rule in the constitution, and we get it back in a single structured-output call.

This mirrors how the `Evaluator-Optimizer` and `Reflection` notebooks in this repo use `with_structured_output` — but the schema here is deliberately "one verdict object per constitutional rule," which is what makes this Constitutional AI rather than generic reflection.


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Define the structured critique schema (one verdict per rule)
# Input   : None
# Output  : `RuleVerdict`, `CritiqueReport` Pydantic models; `critique_model` (structured LLM)
# ─────────────────────────────────────────────

from typing import List

from pydantic import BaseModel, Field


class RuleVerdict(BaseModel):
    """Pass/fail verdict for a single constitutional rule."""

    rule: str = Field(description="The name of the constitutional rule being judged.")
    passes: bool = Field(description="True if the response satisfies this rule, else False.")
    critique: str = Field(
        description=(
            "A short, specific justification for the verdict. If it fails, explain exactly "
            "what in the response violates the rule."
        )
    )


class CritiqueReport(BaseModel):
    """Full critique: one verdict per rule in the constitution."""

    verdicts: List[RuleVerdict] = Field(
        description="One verdict per constitutional rule, in the same order the rules were given."
    )


critique_model = llm.with_structured_output(CritiqueReport)


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Run one structured critique call, judging the response against every rule at once
# Input   : response_text (str), constitution (list of {"name", "rule"})
# Output  : `CritiqueReport` — one RuleVerdict per rule
# ─────────────────────────────────────────────

def critique_response(response_text: str, constitution: list[dict]) -> CritiqueReport:
    """Judge `response_text` against every rule in `constitution`, one verdict per rule."""

    rules_block = "\n".join(
        f"{i + 1}. [{item['name']}] {item['rule']}" for i, item in enumerate(constitution)
    )

    critique_prompt = (
        "You are a strict constitutional reviewer. Judge the RESPONSE below against EACH "
        "rule listed. For every rule, decide whether the response passes or fails, and give "
        "a short, specific critique justifying that verdict. Return exactly one verdict per "
        "rule, in the same order as the rules, using the `rule` field to name which rule "
        "each verdict is for.\n\n"
        f"RULES:\n{rules_block}\n\n"
        f"RESPONSE:\n{response_text}"
    )

    return critique_model.invoke(critique_prompt)


draft_critique = critique_response(draft_response, CONSTITUTION)

print("=== CRITIQUE OF DRAFT (iteration 0) ===")
for verdict in draft_critique.verdicts:
    status = "PASS" if verdict.passes else "FAIL"
    print(f"[{status}] {verdict.rule}: {verdict.critique}")


### Step 4 — Revise Until All Rules Pass (Bounded Loop)

If any rule fails, we feed the current response plus **every failing critique** back to the model and ask for a single rewrite that addresses all of them, then critique the revision again. We cap this at `MAX_ITERATIONS = 2` so the loop always terminates, even if a rule turns out to be unsatisfiable given the prompt.


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Revise a response to address every failing rule's critique
# Input   : response_text (str), failing_verdicts (list[RuleVerdict])
# Output  : revised response text (str)
# ─────────────────────────────────────────────

def revise_response(response_text: str, failing_verdicts: list[RuleVerdict]) -> str:
    """Rewrite `response_text` to address every failing verdict."""

    feedback_block = "\n".join(
        f"- [{v.rule}] {v.critique}" for v in failing_verdicts
    )

    revise_prompt = (
        "Rewrite the RESPONSE below so that it addresses EVERY issue listed in FEEDBACK, "
        "while still answering the original user question. Do not introduce new problems "
        "while fixing these ones.\n\n"
        f"ORIGINAL USER QUESTION:\n{USER_PROMPT}\n\n"
        f"RESPONSE:\n{response_text}\n\n"
        f"FEEDBACK (must all be addressed):\n{feedback_block}"
    )

    return llm.invoke(revise_prompt).content


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Run the bounded critique -> revise loop until all rules pass or the cap is hit
# Input   : draft_response, draft_critique, CONSTITUTION, MAX_ITERATIONS
# Output  : `history` — list of {"iteration", "response", "critique"} across the whole run
# ─────────────────────────────────────────────

MAX_ITERATIONS = 2

current_response = draft_response
current_critique = draft_critique
history = [{"iteration": 0, "response": current_response, "critique": current_critique}]

for iteration in range(1, MAX_ITERATIONS + 1):
    failing = [v for v in current_critique.verdicts if not v.passes]
    if not failing:
        print(f"All rules pass after iteration {iteration - 1}. Stopping.")
        break

    print(f"--- Iteration {iteration}: revising to address {len(failing)} failing rule(s) ---")
    current_response = revise_response(current_response, failing)
    current_critique = critique_response(current_response, CONSTITUTION)
    history.append({"iteration": iteration, "response": current_response, "critique": current_critique})
else:
    remaining = [v.rule for v in current_critique.verdicts if not v.passes]
    if remaining:
        print(f"Reached MAX_ITERATIONS={MAX_ITERATIONS}. Still failing: {remaining}")

final_response = current_response


### Discussion of the Output — Before vs. After, and the Full Verdict Trail

The value of this pattern isn't the final answer alone — it's that we can point at *exactly which rule* changed the response and *why*. Let's print the original draft, the final revised response, and the complete per-rule verdict history across every iteration.


In [ ]:
# ─────────────────────────────────────────────
# Purpose : Show the before/after response and the full per-rule verdict trail
# Input   : history
# Output  : Printed comparison (draft vs. final) and a verdict table per iteration
# ─────────────────────────────────────────────

print("=" * 60)
print("BEFORE (iteration 0 draft)")
print("=" * 60)
print(history[0]["response"])

print("\n" + "=" * 60)
print(f"AFTER (iteration {history[-1]['iteration']} final)")
print("=" * 60)
print(history[-1]["response"])

print("\n" + "=" * 60)
print("FULL VERDICT TRAIL")
print("=" * 60)
for entry in history:
    print(f"\n--- Iteration {entry['iteration']} ---")
    for v in entry["critique"].verdicts:
        status = "PASS" if v.passes else "FAIL"
        print(f"  [{status}] {v.rule}: {v.critique}")


### Key Takeaways

- **Constitutional AI is Reflection with a named, itemized rubric.** Instead of one blended "is this good?" critique, the constitution decomposes judgment into independent, auditable rules — which is what makes per-rule pass/fail (and per-rule revision targeting) possible.
- **Structured output turns critique into data.** Returning a typed `List[RuleVerdict]` (rather than a paragraph of prose) means the loop can programmatically decide which rules failed and feed *only those* critiques into the revision step.
- **Bounding the loop matters.** `MAX_ITERATIONS` guarantees termination even when a rule can't be fully satisfied (e.g. a genuinely ambiguous medical question) — the loop reports what's still failing rather than looping forever.
- **This is a teaching pattern, not a safety system.** A real constitution for production use needs adversarial testing, a calibrated/independent judge, human review, versioning, and monitoring — that belongs in [`12_Production_and_Observability/Safety_and_Alignment/`](../../../12_Production_and_Observability/Safety_and_Alignment/), not in a single notebook's critique loop.
- **Where this composes:** the same "generate → structured per-criterion critique → targeted revise" shape reappears in `05_AI_Agent_Fundamentals/4. Workflow_Pattern/5_Evaluator-optimizer/` and in `18_reflexion`-style notebooks in this folder — Constitutional AI is that shape with the criteria replaced by named principles.
